In [ ]:
# Install necessary libraries
import os
import glob
import numpy as np
import pandas as pd
from xgboost import XGBRegressor
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.model_selection import train_test_split, KFold
import matplotlib.pyplot as plt
import shap

In [ ]:
# Case 1: Precipitation and associated variables alone, and no ETa and SM
feature1 = ['Carbon (0-15 cm)','Carbon (15-30 cm)', 'Clay (0-15 cm)', 'Clay (15-30 cm)','Clay (30-60 cm)', 'Clay (60-90 cm)', 'Clay (90-120 cm)','Sand (0-15 cm)', 
            'Sand (15-30 cm)', 'Sand (30-60 cm)','Sand (60-90 cm)', 'Sand (90-120 cm)', 'Silt (0-15 cm)','Silt (15-30 cm)', 'Silt (30-60 cm)', 'Silt (60-90 cm)',
            'Silt (90-120 cm)', 'OM (0-15 cm)', 'OM (15-30 cm)', 'pH (0-15 cm)', 'pH (15-30 cm)', 'Elevation', 'Aspect', 'Curvature', 'Slope', 'TPI','Nitrogen', 
            'Precipitation']

# Case 2: SM alone and no precipitation and ETa
feature2 = ['Carbon (0-15 cm)','Carbon (15-30 cm)', 'Clay (0-15 cm)', 'Clay (15-30 cm)','Clay (30-60 cm)', 'Clay (60-90 cm)', 'Clay (90-120 cm)','Sand (0-15 cm)', 
            'Sand (15-30 cm)', 'Sand (30-60 cm)','Sand (60-90 cm)', 'Sand (90-120 cm)', 'Silt (0-15 cm)','Silt (15-30 cm)', 'Silt (30-60 cm)', 'Silt (60-90 cm)',
            'Silt (90-120 cm)', 'OM (0-15 cm)', 'OM (15-30 cm)', 'pH (0-15 cm)', 'pH (15-30 cm)', 'Elevation', 'Aspect', 'Curvature', 'Slope', 'TPI','Nitrogen',
            'SM 30 cm', 'SM 60 cm', 'SM 90 cm']
#, 'SM 120 cm', 'SM 150 cm', 'SM 180 cm'

# Case 3: ETa alone and no Precipitation and SM
feature3 = ['Carbon (0-15 cm)','Carbon (15-30 cm)', 'Clay (0-15 cm)', 'Clay (15-30 cm)','Clay (30-60 cm)', 'Clay (60-90 cm)', 'Clay (90-120 cm)','Sand (0-15 cm)', 
            'Sand (15-30 cm)', 'Sand (30-60 cm)','Sand (60-90 cm)', 'Sand (90-120 cm)', 'Silt (0-15 cm)','Silt (15-30 cm)', 'Silt (30-60 cm)', 'Silt (60-90 cm)',
            'Silt (90-120 cm)', 'OM (0-15 cm)', 'OM (15-30 cm)', 'pH (0-15 cm)', 'pH (15-30 cm)', 'Elevation', 'Aspect', 'Curvature', 'Slope', 'TPI','Nitrogen', 
           'ETa']

# Case 4: Precipitation and associated variables alone, and ETa only 
feature4 = ['Carbon (0-15 cm)','Carbon (15-30 cm)', 'Clay (0-15 cm)', 'Clay (15-30 cm)','Clay (30-60 cm)', 'Clay (60-90 cm)', 'Clay (90-120 cm)','Sand (0-15 cm)', 
            'Sand (15-30 cm)', 'Sand (30-60 cm)','Sand (60-90 cm)', 'Sand (90-120 cm)', 'Silt (0-15 cm)','Silt (15-30 cm)', 'Silt (30-60 cm)', 'Silt (60-90 cm)',
            'Silt (90-120 cm)', 'OM (0-15 cm)', 'OM (15-30 cm)', 'pH (0-15 cm)', 'pH (15-30 cm)', 'Elevation', 'Aspect', 'Curvature', 'Slope', 'TPI','Nitrogen', 
           'Precipitation', 'ETa']

# Case 5: Precipitation and associated variables alone, and SM only 
feature5 = ['Carbon (0-15 cm)','Carbon (15-30 cm)', 'Clay (0-15 cm)', 'Clay (15-30 cm)','Clay (30-60 cm)', 'Clay (60-90 cm)', 'Clay (90-120 cm)','Sand (0-15 cm)', 
            'Sand (15-30 cm)', 'Sand (30-60 cm)','Sand (60-90 cm)', 'Sand (90-120 cm)', 'Silt (0-15 cm)','Silt (15-30 cm)', 'Silt (30-60 cm)', 'Silt (60-90 cm)',
            'Silt (90-120 cm)', 'OM (0-15 cm)', 'OM (15-30 cm)', 'pH (0-15 cm)', 'pH (15-30 cm)', 'Elevation', 'Aspect', 'Curvature', 'Slope', 'TPI','Nitrogen', 
            'SM 30 cm', 'SM 60 cm', 'SM 90 cm', 'Precipitation']

# Case 6: SM and ETa only
feature6 = ['Carbon (0-15 cm)','Carbon (15-30 cm)', 'Clay (0-15 cm)', 'Clay (15-30 cm)','Clay (30-60 cm)', 'Clay (60-90 cm)', 'Clay (90-120 cm)','Sand (0-15 cm)', 
            'Sand (15-30 cm)', 'Sand (30-60 cm)','Sand (60-90 cm)', 'Sand (90-120 cm)', 'Silt (0-15 cm)','Silt (15-30 cm)', 'Silt (30-60 cm)', 'Silt (60-90 cm)',
            'Silt (90-120 cm)', 'OM (0-15 cm)', 'OM (15-30 cm)', 'pH (0-15 cm)', 'pH (15-30 cm)', 'Elevation', 'Aspect', 'Curvature', 'Slope', 'TPI','Nitrogen', 
            'SM 30 cm', 'SM 60 cm', 'SM 90 cm', 'ETa']

# Case 7: All
feature7 = ['Carbon (0-15 cm)','Carbon (15-30 cm)', 'Clay (0-15 cm)', 'Clay (15-30 cm)','Clay (30-60 cm)', 'Clay (60-90 cm)', 'Clay (90-120 cm)','Sand (0-15 cm)', 
            'Sand (15-30 cm)', 'Sand (30-60 cm)','Sand (60-90 cm)', 'Sand (90-120 cm)', 'Silt (0-15 cm)','Silt (15-30 cm)', 'Silt (30-60 cm)', 'Silt (60-90 cm)',
            'Silt (90-120 cm)', 'OM (0-15 cm)', 'OM (15-30 cm)', 'pH (0-15 cm)', 'pH (15-30 cm)', 'Elevation', 'Aspect', 'Curvature', 'Slope', 'TPI','Nitrogen', 
            'SM 30 cm', 'SM 60 cm', 'SM 90 cm', 'ETa', 'Precipitation']

# Case 8: All
feature8 = ['SM 30 cm', 'SM 60 cm', 'SM 90 cm', 'ETa', 'Precipitation', 'Nitrogen', 'Elevation', 'Aspect', 'Curvature', 'Slope', 'TPI']

In [ ]:
import os, glob, re
import pandas as pd

BASE = r"./data/processed/Wheat"

YEAR_RE = re.compile(r"(19|20)\d{2}")

def infer_year_from_cols(df: pd.DataFrame) -> pd.Series | None:
    # 1) direct 'year' column (any case)
    year_cols = [c for c in df.columns if c.lower() == "year"]
    if year_cols:
        return df[year_cols[0]].astype(str)

    # 2) any column that looks like a date
    date_cols = [c for c in df.columns if "date" in c.lower()]
    for c in date_cols:
        try:
            return pd.to_datetime(df[c], errors="coerce").dt.year.astype("Int64").astype(str)
        except Exception:
            pass
    return None

def infer_year_from_string(s: str) -> str | None:
    m = YEAR_RE.search(s)
    return m.group(0) if m else None

def load_folder(folder_path: str):
    files = glob.glob(os.path.join(folder_path, "*.csv"))
    frames = []
    for f in files:
        df = pd.read_csv(f)
        df = df.assign(
            Field=os.path.splitext(os.path.basename(f))[0],
            SourceFile=os.path.basename(f),
            SourcePath=f
        )
        yr = infer_year_from_cols(df)

        if yr is None:
            # 3) try file name, then full path, then 'Field' text
            yr = infer_year_from_string(df["SourceFile"].iloc[0]) \
                 or infer_year_from_string(df["SourcePath"].iloc[0]) \
                 or infer_year_from_string(df["Field"].iloc[0])

            if yr is not None:
                df["Year"] = str(yr)
            else:
                # leave as missing for now; caller can decide
                df["Year"] = pd.NA
        else:
            df["Year"] = yr.astype(str)

        frames.append(df)

    out = pd.concat(frames, ignore_index=True)
    out["Year"] = out["Year"].astype("string")
    return out

# Years
years = ["2019", "2020", "2021", "2022", "2023", "2024"]
dfs_year = {y: load_folder(os.path.join(BASE, y)) for y in years}

# Combine subfolders (may mix years; we infer from file names/paths/columns)
df_ASP = load_folder(os.path.join(BASE, "Combine_Add", "ASP"))
df_BAU = load_folder(os.path.join(BASE, "Combine_Add", "BAU"))
df_ASP_BAU = pd.concat( [df_ASP, df_BAU], ignore_index=True)

# (Optional) access as variables like your example:
df_2019, df_2020, df_2021, df_2022, df_2023, df_2024 = (dfs_year[y] for y in years)

# Define wet/dry year sets
NORMAL_YEARS = {"2019", "2021", "2024"}
DRY_YEARS = {"2020", "2022"}
WET_YEARS = {"2023"}


# Filter (drop rows where Year couldn't be inferred)
df_wet_ASP = df_ASP[df_ASP["Year"].isin(WET_YEARS)].copy()
df_wet_BAU = df_BAU[df_BAU["Year"].isin(WET_YEARS)].copy()
df_normal_BAU = df_BAU[df_BAU["Year"].isin(NORMAL_YEARS)].copy()
df_normal_ASP = df_ASP[df_ASP["Year"].isin(NORMAL_YEARS)].copy()
df_dry_ASP = df_ASP[df_ASP["Year"].isin(DRY_YEARS)].copy()
df_dry_BAU = df_BAU[df_BAU["Year"].isin(DRY_YEARS)].copy()
df_wet_ASP_BAU = df_ASP_BAU[df_ASP_BAU["Year"].isin(WET_YEARS)].copy()
df_dry_ASP_BAU = df_ASP_BAU[df_ASP_BAU["Year"].isin(DRY_YEARS)].copy()
df_normal_ASP_BAU = df_ASP_BAU[df_ASP_BAU["Year"].isin(NORMAL_YEARS)].copy()

# Optional: see if any rows lacked a year so you can investigate
missing_year_ASP = df_ASP["Year"].isna().sum()
missing_year_BAU = df_BAU["Year"].isna().sum()
print("Missing Year rows -> ASP:", missing_year_ASP, "BAU:", missing_year_BAU)

# (Optional) quick check
for name, df in {
    **{f"df_{y}": dfs_year[y] for y in years},
    "df_ASP": df_ASP, 
    "df_BAU": df_BAU,
    "df_ASP_BAU": df_ASP_BAU,
    "df_dry_BAU": df_dry_BAU, "df_wet_BAU": df_wet_BAU, "df_normal_BAU": df_normal_BAU,
    "df_dry_ASP": df_dry_ASP, "df_wet_ASP": df_wet_ASP, "df_normal_ASP": df_normal_ASP,
    "df_dry_ASP_BAU": df_dry_ASP_BAU, 
    "df_wet_ASP_BAU": df_wet_ASP_BAU,
    "df_normal_ASP_BAU": df_normal_ASP_BAU
}.items():
    print(name, df.shape)

In [ ]:
feature_sets = {
    "Case1": feature1,
    "Case2": feature2,
    "Case3": feature3,
    "Case4": feature4,
    "Case5": feature5,
    "Case6": feature6,
    "Case7": feature7,
    "Case8": feature8,
}

dataframes = {
    "2019": df_2019,
    "2020": df_2020,
    "2021": df_2021,
    "2022": df_2022,
    "2023": df_2023,
    "2024": df_2024,
    "ASP_All": df_ASP,
    "BAU_All": df_BAU,
    "Wet_ASP": df_wet_ASP,
    "Wet_BAU": df_wet_BAU,
    "Dry_ASP": df_dry_ASP,
    "Dry_BAU": df_dry_BAU,
    "Normal_ASP": df_normal_ASP,
    "Normal_BAU": df_normal_BAU,
    "ASP_BAU_All": df_ASP_BAU,
    "ASP_BAU_All_Dry": df_dry_ASP_BAU,
    "ASP_BAU_All_Wet": df_wet_ASP_BAU,
    "ASP_BAU_All_Normal": df_normal_ASP_BAU,
}

from sklearn.metrics import r2_score, mean_squared_error
import numpy as np

def calc_metrics(y_true, y_pred):
    """
    Returns:
        r2     : R^2
        rmse   : Root Mean Square Error
        rrmse  : Relative RMSE in %
    """
    r2 = r2_score(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    rrmse = (rmse / np.mean(y_true)) * 100
    return r2, rmse, rrmse

## Model for whole years ASP and BAU fields

In [ ]:
target = 'yield'
Y_MULT = 62.77

# Combine results for all dataframes into one CSV
all_results = []

for df_name, df in dataframes.items():
    print(f"\n===== Processing {df_name} =====")

    for name, features in feature_sets.items():
        X = df[features].values
        y = df[target].values * Y_MULT

        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.25, random_state=42
        )

        kf = KFold(n_splits=5, shuffle=True, random_state=42)
        r2_list, rmse_list, rrmse_list = [], [], []

        for tr_idx, val_idx in kf.split(X_train):
            X_tr, X_val = X_train[tr_idx], X_train[val_idx]
            y_tr, y_val = y_train[tr_idx], y_train[val_idx]

            model = XGBRegressor(objective='reg:squarederror', random_state=42)
            model.fit(X_tr, y_tr)
            y_pred_val = model.predict(X_val)
            r2, rmse, rrmse = calc_metrics(y_val, y_pred_val)
            r2_list.append(r2)
            rmse_list.append(rmse)
            rrmse_list.append(rrmse)

        # mean CV results
        r2_cv, rmse_cv, rrmse_cv = np.mean(r2_list), np.mean(rmse_list), np.mean(rrmse_list)

        # test results
        model_final = XGBRegressor(objective='reg:squarederror', random_state=42)
        model_final.fit(X_train, y_train)
        y_pred_test = model_final.predict(X_test)
        r2_test, rmse_test, rrmse_test = calc_metrics(y_test, y_pred_test)

        # append to master list
        all_results.append({
            "DataFrame": df_name,
            "Case": name,
            "R2_CV": r2_cv, "RMSE_CV": rmse_cv, "RRMSE_CV(%)": rrmse_cv,
            "R2_Test": r2_test, "RMSE_Test": rmse_test, "RRMSE_Test(%)": rrmse_test
        })


# Export combined results
final_results_df = pd.DataFrame(all_results)
print(final_results_df.round(3))

# Save to CSV
final_results_df.to_csv(r"./results/All_Years_and_Dry_and_Normal_Years.csv", index=False)

# SHAP Analysis

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import shap
from matplotlib.gridspec import GridSpec, GridSpecFromSubplotSpec
from xgboost import XGBRegressor

# CONFIG
target = "yield"
Y_MULT = 62.77
MODEL_KW = dict(n_estimators=800, max_depth=3, learning_rate=0.01, random_state=42)
TOP_N = 10
OUTDIR = r"./results/plots"
os.makedirs(OUTDIR, exist_ok=True)

# Build dataset dictionary
datasets = {
    "BAU": df_BAU,
    "ASP": df_ASP
}

# Feature set definitions
feature_sets = {
    "Case 1: Precipitation only": feature1,
    "Case 2: SM only": feature2,
    "Case 3: ETa only": feature3,
    "Case 4: Precipitation and ETa": feature4,
    "Case 5: Precipitation and SM": feature5,
    "Case 6: SM and ETa": feature6,
    "Case 7: Precipitation, SM, and ETa": feature7,
    "Case 8: Without soil properties": feature8
}

# Plot helpers
def plot_bar_and_beeswarm(ax_bar, ax_bee, model, X, top_n):
    # Create SHAP values using the fitted model
    explainer = shap.Explainer(model)
    sv = explainer(X)

    mean_abs = np.mean(np.abs(sv.values), axis=0)
    order = np.argsort(mean_abs)[::-1]
    feat_names = X.columns.to_numpy()

    idx = order[:top_n]
    top_feats = feat_names[idx]
    top_vals = mean_abs[idx]

    # ---- Left: bar importance
    bars = ax_bar.barh(top_feats, top_vals)
    ax_bar.invert_yaxis()
    ax_bar.set_xlabel("mean(|SHAP value|)", fontsize=9)
    ax_bar.tick_params(axis='y', labelsize=8)
    ax_bar.spines['top'].set_visible(False)
    ax_bar.spines['right'].set_visible(False)
    ax_bar.tick_params(left=False)

    for i, b in enumerate(bars):
        x = b.get_width()
        y = b.get_y() + b.get_height()/2
        ax_bar.text(x + 0.0005, y, f"{top_vals[i]:.1f}", va='center', ha='left', fontsize=9)

    # ---- Right: beeswarm (top_n features only)
    shap.plots.beeswarm(
        sv[:, idx],
        max_display=top_n,
        show=False,
        plot_size=None,
        ax=ax_bee
    )
    ax_bee.set_xlabel("SHAP value (impact on model output)", fontsize=9)
    ax_bee.set_ylabel("")
    ax_bee.set_yticklabels([])
    ax_bee.tick_params(axis='y', left=False, labelleft=False)

def shap_grid_3x2(ds_name, df, feature_sets, save=True):
    fig = plt.figure(figsize=(15, 15))
    gs = GridSpec(4, 2, figure=fig, wspace=0.30, hspace=0.30)

    cases = list(feature_sets.items())
    for i, (case_name, feats) in enumerate(cases):
        r, c = divmod(i, 2)
        cell = gs[r, c]

        subgs = GridSpecFromSubplotSpec(1, 2, subplot_spec=cell, width_ratios=[1, 2], wspace=0.30)
        ax_bar = fig.add_subplot(subgs[0, 0])
        ax_bee = fig.add_subplot(subgs[0, 1])

        # ✅ Robust to missing features (e.g., 'Nitrogen' removed in BAU)
        feats = [f for f in feats if f in df.columns]
        if not feats:
            ax_bar.axis("off"); ax_bee.axis("off")
            continue

        X = df[feats].copy()
        y = (df[target] * Y_MULT).copy()

        model = XGBRegressor(**MODEL_KW)
        model.fit(X, y)

        plot_bar_and_beeswarm(ax_bar, ax_bee, model, X, TOP_N)

        # Bold per-subplot title
        ax_bar.set_title(case_name, fontweight="bold", loc="center")

    # Layout and save
    fig.tight_layout(rect=[0, 0, 1, 0.97])
    if save:
        outpath = os.path.join(OUTDIR, f"SHAP_{ds_name}.pdf")
        fig.savefig(outpath, dpi=400, bbox_inches="tight")
        print(f"[OK] saved: {outpath}")
    plt.close(fig)

# Run for all sets
for name, df in datasets.items():
    shap_grid_3x2(name, df, feature_sets, save=True)

## Temporal Blocking

In [ ]:
# CONFIG
import re
import numpy as np
import pandas as pd
from sklearn.metrics import r2_score, mean_squared_error
from xgboost import XGBRegressor

# -DataFrame to use
df = df_ASP_BAU.copy()

# Target & scaling
target = "yield"   
Y_MULT = 62.77     

# Feature groups 
feature_sets = {
    "Case1": feature1,
    "Case2": feature2,
    "Case3": feature3,
    "Case4": feature4,
    "Case5": feature5,
    "Case6": feature6,
    "Case7": feature7,
    "Case8": feature8,
}

# XGB params
xgb_params = dict(
    n_estimators=600,
    max_depth=4,
    learning_rate=0.01,
    subsample=0.9,
    colsample_bytree=0.9,
    min_child_weight=1.0,
    reg_lambda=1.0,
    objective="reg:squarederror",
    random_state=42,
    n_jobs=-1,
)

#PREPARE COLUMNS
# Parse "Field" like "S2_2022_Wheat" -> FieldName="S2", Year="2022"
def parse_field_year(s: str):
    parts = str(s).split("_")
    fld = parts[0] if len(parts) > 0 else np.nan
    yr  = parts[1] if len(parts) > 1 and re.fullmatch(r"\d{4}", parts[1]) else np.nan
    return pd.Series({"FieldName": fld, "Year": yr})

# Create FieldName and Year columns
df[["FieldName", "Year"]] = df["Field"].apply(parse_field_year)

# Ensure Year is string to avoid numeric casting issues in grouping/printing
df["Year"] = df["Year"].astype(str)

# Make the scaled target for evaluation (keeps original intact)
df["_y_scaled"] = df[target] * Y_MULT

# =================== HELPER FUNCTIONS ===================
def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

def rrmse_percent(y_true, y_pred):
    return rmse(y_true, y_pred) / np.maximum(1e-12, np.mean(y_true)) * 100.0

def safe_subset(df_in: pd.DataFrame, cols: list):
    # keep only columns that actually exist (and warn if missing)
    have = [c for c in cols if c in df_in.columns]
    missing = sorted(set(cols) - set(have))
    if missing:
        print(f"[WARN] {len(missing)} feature(s) missing and will be ignored:\n  {missing[:10]}{' ...' if len(missing)>10 else ''}")
    return have

def group_loocv(df_in: pd.DataFrame, group_col: str, feature_sets: dict, xgb_params: dict):
    """
    Generic LOOCV over 'group_col' (here we will use 'Year').
    Returns a DataFrame of metrics per feature set.
    """
    groups = df_in[group_col].dropna().unique().tolist()
    results = {name: {"y_true": [], "y_pred": []} for name in feature_sets}

    for g in groups:
        train_idx = df_in[group_col] != g
        test_idx  = df_in[group_col] == g

        df_tr = df_in.loc[train_idx].copy()
        df_te = df_in.loc[test_idx].copy()

        for set_name, cols in feature_sets.items():
            use_cols = safe_subset(df_in, cols)
            if len(use_cols) == 0:
                continue

            # Drop rows with NA in X or y for this fold
            tr = df_tr.dropna(subset=use_cols + ["_y_scaled"])
            te = df_te.dropna(subset=use_cols + ["_y_scaled"])
            if tr.empty or te.empty:
                continue

            X_tr, y_tr = tr[use_cols].values, tr["_y_scaled"].values
            X_te, y_te = te[use_cols].values, te["_y_scaled"].values

            model = XGBRegressor(**xgb_params)
            model.fit(X_tr, y_tr)
            y_hat = model.predict(X_te)

            results[set_name]["y_true"].append(y_te)
            results[set_name]["y_pred"].append(y_hat)

    # Aggregate metrics per feature set
    rows = []
    for set_name, d in results.items():
        if len(d["y_true"]) == 0:
            rows.append([set_name, np.nan, np.nan, np.nan])
            continue
        y_true_all = np.concatenate(d["y_true"])
        y_pred_all = np.concatenate(d["y_pred"])
        r2   = r2_score(y_true_all, y_pred_all)
        e    = rmse(y_true_all, y_pred_all)
        rr   = rrmse_percent(y_true_all, y_pred_all)
        rows.append([set_name, r2, e, rr])

    out = pd.DataFrame(rows, columns=["Feature_Set", "R2", "RMSE", "RRMSE_%"])
    # Sort by the numeric part of the case label if present
    out = out.sort_values("Feature_Set", key=lambda s: s.str.extract(r"(\d+)").astype(float).fillna(0)[0])
    return out.reset_index(drop=True)

# TEMPORAL CV ONLY
print("Unique years :", sorted(df["Year"].dropna().unique().tolist()))

print("\n=== TEMPORAL LOOCV (leave-one-year-out) ===")
temporal_metrics = group_loocv(
    df_in=df,
    group_col="Year",
    feature_sets=feature_sets,
    xgb_params=xgb_params
)
print(temporal_metrics.to_string(index=False))

## Spatial Blocking

In [ ]:
# CONFIG
import re
import numpy as np
import pandas as pd
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.model_selection import RandomizedSearchCV, KFold, train_test_split
from scipy.stats import randint as sp_randint, uniform as sp_uniform
from xgboost import XGBRegressor

# DataFrame to use
df =df_ASP.copy()

# Target & scaling
target = "yield"
Y_MULT = 62.77

# Feature groups
feature_sets = {
    "Case1": feature1,
    "Case2": feature2,
    "Case3": feature3,
    "Case4": feature4,
    "Case5": feature5,
    "Case6": feature6,
    "Case7": feature7,
    "Case8": feature8,
}

# Base XGB params
base_xgb_params = dict(
    objective="reg:squarederror",
    n_estimators=500,         
    learning_rate=0.05,
    max_depth=5,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=3.0,
    reg_lambda=1.0,
    reg_alpha=0.0,
    random_state=42,
    n_jobs=-1,
    tree_method="hist",
    eval_metric="rmse",        
)

# PREPARE COLUMNS
def parse_field_year(s: str):
    parts = str(s).split("_")
    fld = parts[0] if len(parts) > 0 else np.nan
    yr  = parts[1] if len(parts) > 1 and re.fullmatch(r"\d{4}", parts[1]) else np.nan
    return pd.Series({"FieldName": fld, "Year": yr})

if "FieldName" not in df.columns or "Year" not in df.columns:
    df[["FieldName", "Year"]] = df["Field"].apply(parse_field_year)

df["Year"] = df["Year"].astype(str)
df["_y_scaled"] = df[target] * Y_MULT
df["Year_num"] = pd.to_numeric(df["Year"], errors="coerce")

# HELPERS 
def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

def rrmse_percent(y_true, y_pred):
    return rmse(y_true, y_pred) / max(1e-12, np.mean(y_true)) * 100.0

def safe_subset(df_in: pd.DataFrame, cols: list):
    have = [c for c in cols if c in df_in.columns]
    missing = sorted(set(cols) - set(have))
    if missing:
        print(f"[WARN] {len(missing)} feature(s) missing and will be ignored:\n  {missing[:10]}{' ...' if len(missing)>10 else ''}")
    return have

# Per-year scalers (train-only)
def compute_group_scalers(df_train: pd.DataFrame, cols: list, group_col: str = "Year"):
    scalers = {}
    for g, d in df_train.groupby(group_col):
        means = d[cols].mean(numeric_only=True)
        stds  = d[cols].std(ddof=0, numeric_only=True).replace(0, 1.0)
        scalers[g] = (means, stds)
    global_means = df_train[cols].mean(numeric_only=True)
    global_stds  = df_train[cols].std(ddof=0, numeric_only=True).replace(0, 1.0)
    return scalers, (global_means, global_stds)

def apply_group_scalers(df_in: pd.DataFrame, cols: list, scalers, global_scaler, group_col: str = "Year"):
    out = df_in.copy()
    g_means, g_stds = global_scaler
    out[cols] = (out[cols] - g_means) / g_stds
    for g, (m, s) in scalers.items():
        mask = out[group_col] == g
        if mask.any():
            out.loc[mask, cols] = (out.loc[mask, cols] - m) / s
    return out

# Randomized tuning + early stopping (compatible with older xgboost)
def tune_and_fit_xgb(X_tr, y_tr, X_val, y_val, base_params: dict, n_iter: int = 12):
    param_dist = {
        "max_depth": sp_randint(3, 8),
        "learning_rate": sp_uniform(0.02, 0.12),
        "subsample": sp_uniform(0.6, 0.4),
        "colsample_bytree": sp_uniform(0.6, 0.4),
        "min_child_weight": sp_uniform(1.0, 8.0),
        "reg_lambda": sp_uniform(0.0, 5.0),
        "reg_alpha": sp_uniform(0.0, 1.0),
        "n_estimators": sp_randint(500, 1500),
    }

    base = XGBRegressor(**base_params)
    rs = RandomizedSearchCV(
        estimator=base,
        param_distributions=param_dist,
        n_iter=n_iter,
        scoring="neg_root_mean_squared_error",
        cv=KFold(n_splits=3, shuffle=True, random_state=42),
        refit=False,
        random_state=42,
        n_jobs=-1,
        verbose=0,
    )
    try:
        rs.fit(X_tr, y_tr)
        best_params = {**base_params, **rs.best_params_}
    except Exception as e:
        print(f"[WARN] Hyperparameter search skipped due to: {e}")
        best_params = base_params

    model = XGBRegressor(**best_params)
    try:
        model.fit(
            X_tr, y_tr,
            eval_set=[(X_val, y_val)],
            verbose=False,
            early_stopping_rounds=50
        )
    except TypeError:
        model.fit(X_tr, y_tr, verbose=False)
    return model

# LOOCV with scaling + tuning (generic; we'll use FieldName only)
def group_loocv(df_in: pd.DataFrame,
                group_col: str,
                feature_sets: dict,
                base_xgb_params: dict,
                include_year_feature: bool = True,
                scale_per_year: bool = True):
    groups = df_in[group_col].dropna().unique().tolist()
    results = {name: {"y_true": [], "y_pred": []} for name in feature_sets}

    for g in groups:
        train_idx = df_in[group_col] != g
        test_idx  = df_in[group_col] == g

        df_tr = df_in.loc[train_idx].copy()
        df_te = df_in.loc[test_idx].copy()

        for set_name, cols in feature_sets.items():
            use_cols = safe_subset(df_in, cols)
            if include_year_feature and "Year_num" in df_in.columns:
                use_cols = use_cols + ["Year_num"]

            tr = df_tr.dropna(subset=use_cols + ["_y_scaled"])
            te = df_te.dropna(subset=use_cols + ["_y_scaled"])
            if tr.empty or te.empty:
                continue

            if scale_per_year and "Year" in df_in.columns:
                feat_cols = [c for c in use_cols if c != "Year_num"]
                scalers, global_scaler = compute_group_scalers(
                    tr.assign(Year=tr["Year"].astype(str)), cols=feat_cols, group_col="Year"
                )
                X_tr = apply_group_scalers(
                    tr.assign(Year=tr["Year"].astype(str)), cols=feat_cols,
                    scalers=scalers, global_scaler=global_scaler, group_col="Year"
                )[use_cols]
                X_te = apply_group_scalers(
                    te.assign(Year=te["Year"].astype(str)), cols=feat_cols,
                    scalers=scalers, global_scaler=global_scaler, group_col="Year"
                )[use_cols]
            else:
                X_tr, X_te = tr[use_cols], te[use_cols]

            y_tr, y_te = tr["_y_scaled"].values, te["_y_scaled"].values

            X_tr2, X_val, y_tr2, y_val = train_test_split(
                X_tr, y_tr, test_size=0.2, random_state=42
            )

            model = tune_and_fit_xgb(X_tr2, y_tr2, X_val, y_val, base_xgb_params, n_iter=10)
            y_hat = model.predict(X_te)

            results[set_name]["y_true"].append(y_te)
            results[set_name]["y_pred"].append(y_hat)

    rows = []
    for set_name, d in results.items():
        if len(d["y_true"]) == 0:
            rows.append([set_name, np.nan, np.nan, np.nan])
            continue
        y_true_all = np.concatenate(d["y_true"])
        y_pred_all = np.concatenate(d["y_pred"])
        r2 = r2_score(y_true_all, y_pred_all)
        e  = rmse(y_true_all, y_pred_all)
        rr = rrmse_percent(y_true_all, y_pred_all)
        rows.append([set_name, r2, e, rr])

    out = pd.DataFrame(rows, columns=["Feature_Set", "R2", "RMSE", "RRMSE_%"])
    out = out.sort_values("Feature_Set", key=lambda s: s.str.extract(r"(\d+)").astype(float).fillna(0)[0])
    return out.reset_index(drop=True)

# =================== RUN SPATIAL CV & PRINT ===================
print("Unique fields:", sorted(df["FieldName"].dropna().unique().tolist()))
print("Unique years :", sorted(df["Year"].dropna().unique().tolist()))

print("\n=== SPATIAL LOOCV (leave-one-field-out) ===")
spatial_metrics = group_loocv(
    df, group_col="FieldName",
    feature_sets=feature_sets,
    base_xgb_params=base_xgb_params,
    include_year_feature=True,
    scale_per_year=True
)
print(spatial_metrics.to_string(index=False))